**Description**: This script takes the output files from an XGBoost model and a LightGBM model and combines them into a final submission file.

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/prev-submissions/submission_lgbm.csv
/kaggle/input/prev-submissions/submission_xgb.csv
/kaggle/input/antimicrobial-resistance-prediction-from-maldi-tof/sample_submission.csv
/kaggle/input/antimicrobial-resistance-prediction-from-maldi-tof/species_mapping.csv
/kaggle/input/antimicrobial-resistance-prediction-from-maldi-tof/train.csv
/kaggle/input/antimicrobial-resistance-prediction-from-maldi-tof/test.csv


Spectrum (fixed m/z positions)
   ↓
LocallyConnected1D (NO invariance)
   ↓
Species embedding
   ↓
Shared encoder
   ↓
Antibiotic-specific heads (8 modelos)


In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import os

from sklearn.preprocessing import Normalizer, LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score


In [3]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

ID_COL = "sample_id"
SPECIES_COL = "species_id"

ANTIBIOTICS = [
    "Ampicillin", "Levofloxacin", "Ciprofloxacin", "Imipenem",
    "Amoxicillin_Clavulanic_acid", "Ertapenem", "Cefotaxime", "Cefuroxime"
]

EPOCHS_PRETRAIN = 25
EPOCHS_FINETUNE = 20
BATCH_SIZE = 64
LR = 1e-3

CHECKPOINT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)


In [4]:
train = pd.read_csv("/kaggle/input/antimicrobial-resistance-prediction-from-maldi-tof/train.csv")
test  = pd.read_csv("/kaggle/input/antimicrobial-resistance-prediction-from-maldi-tof/test.csv")

SPECTRUM_COLS = [c for c in train.columns if c not in [ID_COL, SPECIES_COL] + ANTIBIOTICS]

# Normalize spectra (cosine-like)
norm = Normalizer()
train[SPECTRUM_COLS] = norm.fit_transform(train[SPECTRUM_COLS].fillna(0))
test[SPECTRUM_COLS]  = norm.transform(test[SPECTRUM_COLS].fillna(0))

# Encode species
le = LabelEncoder()
train["species_enc"] = le.fit_transform(train[SPECIES_COL])
test["species_enc"]  = le.transform(test[SPECIES_COL])

SPECTRUM_LEN = len(SPECTRUM_COLS)
N_SPECIES = train["species_enc"].nunique()


/tmp/ipykernel_23/1431283195.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train["species_enc"] = le.fit_transform(train[SPECIES_COL])
/tmp/ipykernel_23/1431283195.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test["species_enc"]  = le.transform(test[SPECIES_COL])


In [5]:
class MALDIDataset(torch.utils.data.Dataset):
    def __init__(self, df, antibiotic=None):
        self.X = df[SPECTRUM_COLS].values.astype(np.float32)
        self.sp = df["species_enc"].values.astype(np.int64)
        self.y = None
        if antibiotic is not None:
            self.y = df[antibiotic].values.astype(np.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = torch.tensor(self.X[idx]).unsqueeze(0)
        sp = torch.tensor(self.sp[idx])
        if self.y is None:
            return x, sp
        y = torch.tensor(self.y[idx])
        return x, sp, y


In [6]:
class LocallyConnected1D(nn.Module):
    def __init__(self, in_channels, out_channels, length):
        super().__init__()
        self.weight = nn.Parameter(
            torch.randn(out_channels, in_channels, length)
        )
        self.bias = nn.Parameter(torch.zeros(out_channels, length))

    def forward(self, x):
        # x: [B, C, L]
        return (x.unsqueeze(1) * self.weight.unsqueeze(0)).sum(2) + self.bias.unsqueeze(0)


In [7]:
class MALDIEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.local = LocallyConnected1D(1, 32, SPECTRUM_LEN)
        self.bn = nn.BatchNorm1d(32)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.local(x)
        x = self.bn(x)
        return self.relu(x)


In [8]:
def pretrain_encoder(encoder, df):
    encoder.train()
    opt = optim.Adam(encoder.parameters(), lr=LR)
    loss_fn = nn.MSELoss()

    X = torch.tensor(df[SPECTRUM_COLS].values, dtype=torch.float32).unsqueeze(1)

    for epoch in range(EPOCHS_PRETRAIN):
        idx = torch.randperm(len(X))
        Xb = X[idx][:BATCH_SIZE].to(DEVICE)

        mask = torch.rand_like(Xb) > 0.15
        X_masked = Xb * mask

        out = encoder(X_masked)
        loss = loss_fn(out, encoder(Xb).detach())

        opt.zero_grad()
        loss.backward()
        opt.step()

        torch.save(encoder.state_dict(), f"{CHECKPOINT_DIR}/encoder_pretrain_epoch{epoch}.pt")

    print("SSL pretraining done")


In [9]:
class AntibioticModel(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.sp_emb = nn.Embedding(N_SPECIES, 8)
        self.fc = nn.Linear(32 * SPECTRUM_LEN + 8, 1)

    def forward(self, x, sp):
        x = self.encoder(x).view(x.size(0), -1)
        sp = self.sp_emb(sp)
        x = torch.cat([x, sp], dim=1)
        return self.fc(x)


In [10]:
test_preds = np.zeros((len(test), len(ANTIBIOTICS)))

encoder = MALDIEncoder().to(DEVICE)
pretrain_encoder(encoder, train)

for i, ab in enumerate(ANTIBIOTICS):
    print(f"\nTraining {ab}")

    mask = train[ab].notna()
    df_ab = train.loc[mask]

    skf = StratifiedKFold(5, shuffle=True, random_state=42)
    oof_test = np.zeros(len(test))

    for fold, (tr, va) in enumerate(skf.split(df_ab, df_ab[SPECIES_COL])):

        model = AntibioticModel(encoder).to(DEVICE)
        opt = optim.Adam(model.parameters(), lr=LR)
        loss_fn = nn.BCEWithLogitsLoss()

        train_ds = MALDIDataset(df_ab.iloc[tr], ab)
        test_ds  = MALDIDataset(test)

        train_loader = torch.utils.data.DataLoader(
            train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False
        )
        test_loader = torch.utils.data.DataLoader(
            test_ds, batch_size=256
        )

        # -------- TRAIN --------
        for epoch in range(EPOCHS_FINETUNE):
            model.train()
            for x, sp, y in train_loader:
                x = x.to(DEVICE)
                sp = sp.to(DEVICE)
                y = y.to(DEVICE)

                logits = model(x, sp).view(-1)   # ✅ FIX
                loss = loss_fn(logits, y)

                opt.zero_grad()
                loss.backward()
                opt.step()

        # -------- TEST PRED --------
        model.eval()
        preds = []
        with torch.no_grad():
            for x, sp in test_loader:
                x = x.to(DEVICE)
                sp = sp.to(DEVICE)
                logits = model(x, sp).view(-1)   # ✅ FIX
                preds.append(torch.sigmoid(logits).cpu().numpy())

        oof_test += np.concatenate(preds) / 5

    test_preds[:, i] = oof_test



SSL pretraining done

Training Ampicillin

Training Levofloxacin

Training Ciprofloxacin

Training Imipenem

Training Amoxicillin_Clavulanic_acid

Training Ertapenem

Training Cefotaxime

Training Cefuroxime


In [11]:
submission = pd.DataFrame(test_preds, columns=ANTIBIOTICS)
submission.insert(0, ID_COL, test[ID_COL].values)
submission.to_csv("submission.csv", index=False)

print("submission.csv generated")


submission.csv generated
